The goal is to add stop data to the raw data files. It then will get processed by the cleaning data to give us a complete look at all the info we have.

In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import os

In [2]:
def stopId_extract(route: str) -> dict:
    filehandle = open(f'{route}_stoplist.html')
    link_pattern = re.compile(r'^stophistory')
    soup = BeautifulSoup(markup= filehandle, features ='html.parser')
    links = soup.find_all('a', href=link_pattern)

    stop_link_dict = {}
    for idx, link in enumerate(links):
        # We keep idx here to make sure the stops are being generated in
        # the correct order.
        stop_link_dict[idx] = link.text

    return stop_link_dict

In [3]:
stop_list = list(stopId_extract('506').values())

In [4]:
stop_list[:5]

['Main Street Station at Bay 7',
 'Main St at Gerrard St East',
 'Norwood Rd',
 'Glenmount Park Rd',
 'Golfview Ave']

In [5]:
# Load the raw data
raw_data_path = '../data/raw_data/schedule_data'
dataframes = [pd.read_csv(os.path.join(raw_data_path,x)) for x in os.listdir(raw_data_path) if x[-4:] == '.csv']

raw_df = pd.concat(dataframes, ignore_index=True)

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_70999/3688239977.py:3: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(os.path.join(raw_data_path,x)) for x in os.listdir(raw_data_path) if x[-4:] == '.csv']
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_70999/3688239977.py:3: DtypeWarning: Columns (3,9) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(os.path.join(raw_data_path,x)) for x in os.listdir(raw_data_path) if x[-4:] == '.csv']


In [6]:
raw_df.head()

,Unnamed: 0,Time,Vehicle,Gap,Headway,Schedule,Destination,day,day of the week,Riders after stop
0,0,7:50:00AM (sched),4574.0,NaN,NaN,Scheduled at 7:50:00AM,East to Main Street Station,2025-01-01,2,NaN
1,1,7:55:00AM (sched),4570.0,NaN,5:00,Scheduled at 7:55:00AM,East to Main Street Station,2025-01-01,2,NaN
2,2,7:55:00AM (sched),4574.0,NaN,NaN,Scheduled at 7:55:00AM,West to High Park,2025-01-01,2,NaN
3,3,8:03:00AM (sched),4570.0,NaN,8:00,Scheduled at 8:03:00AM,West to High Park,2025-01-01,2,NaN
4,4,8:05:00AM (sched),4577.0,NaN,10:00,Scheduled at 8:05:00AM,East to Main Street Station,2025-01-01,2,NaN


In [7]:
raw_df['day'] =raw_df['day'].apply(pd.Timestamp)

In [8]:
raw_df_E = raw_df[raw_df['Destination'].str[0] == 'E']
raw_df_W = raw_df[raw_df['Destination'].str[0] == 'W']

In [8]:
raw_df_E = raw_df[raw_df['Destination'].str[0] == 'E']
raw_df_W = raw_df[raw_df['Destination'].str[0] == 'W']

In [9]:
raw_df_E[:10]

,Unnamed: 0,Time,Vehicle,Gap,Headway,Schedule,Destination,day,day of the week,Riders after stop
0,0,7:50:00AM (sched),4574.0,NaN,NaN,Scheduled at 7:50:00AM,East to Main Street Station,2025-01-01,2,NaN
1,1,7:55:00AM (sched),4570.0,NaN,5:00,Scheduled at 7:55:00AM,East to Main Street Station,2025-01-01,2,NaN
4,4,8:05:00AM (sched),4577.0,NaN,10:00,Scheduled at 8:05:00AM,East to Main Street Station,2025-01-01,2,NaN
6,6,8:15:00AM (sched),4400.0,NaN,10:00,Scheduled at 8:15:00AM,East to Main Street Station,2025-01-01,2,NaN
8,8,8:25:00AM (sched),4481.0,NaN,10:00,Scheduled at 8:25:00AM,East to Main Street Station,2025-01-01,2,NaN
10,10,8:34:00AM (sched),4511.0,NaN,9:00,Scheduled at 8:34:00AM,East to Main Street Station,2025-01-01,2,NaN
12,12,8:44:00AM (sched),4552.0,NaN,10:00,Scheduled at 8:44:00AM,East to Main Street Station,2025-01-01,2,NaN
14,14,8:54:00AM (sched),4575.0,NaN,10:00,Scheduled at 8:54:00AM,East to Main Street Station,2025-01-01,2,NaN
16,16,9:07:00AM (sched),4591.0,NaN,13:00,Scheduled at 9:07:00AM,East to Main Street Station,2025-01-01,2,NaN
18,18,9:17:00AM (sched),4503.0,NaN,10:00,Scheduled at 9:17:00AM,East to Main Street Station,2025-01-01,2,NaN


In [10]:
# we'll treat 'stop list' and the rows needed to be added as a stack.
stop_list = list(stopId_extract('506').values())
stops_by_date = []
rows_to_add = list(range(raw_df_E.shape[0]))
current_row = 0
while stop_list != []:
    current_stop = stop_list.pop(0)
    prev_row = current_row
    while rows_to_add != []:
        current_row = rows_to_add[0]
        if raw_df_E['day'].iloc[current_row] >= raw_df_E['day'].iloc[prev_row]:
            prev_row = current_row
            rows_to_add.pop(0)
            stops_by_date.append(current_stop)
        else:
            break

len(stops_by_date)


2563667

In [11]:
# Check we got the right number.
len(stops_by_date) - raw_df_E.shape[0]

0

In [12]:
 raw_df_E['stop'] = stops_by_date

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_70999/734796819.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_df_E['stop'] = stops_by_date


In [13]:
raw_df_E.head()

,Unnamed: 0,Time,Vehicle,Gap,Headway,Schedule,Destination,day,day of the week,Riders after stop,stop
0,0,7:50:00AM (sched),4574.0,NaN,NaN,Scheduled at 7:50:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7
1,1,7:55:00AM (sched),4570.0,NaN,5:00,Scheduled at 7:55:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7
4,4,8:05:00AM (sched),4577.0,NaN,10:00,Scheduled at 8:05:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7
6,6,8:15:00AM (sched),4400.0,NaN,10:00,Scheduled at 8:15:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7
8,8,8:25:00AM (sched),4481.0,NaN,10:00,Scheduled at 8:25:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7


In [15]:
# Same thing for westbound, but we reverse the pops.
stop_list = list(stopId_extract('506').values())
stops_by_date = []
rows_to_add = list(range(raw_df_W.shape[0]))
current_row = 0
while stop_list != []:
    current_stop = stop_list.pop()
    prev_row = current_row
    while rows_to_add != []:
        current_row = rows_to_add[0]
        if raw_df_W['day'].iloc[current_row] >= raw_df_W['day'].iloc[prev_row]:
            prev_row = current_row
            rows_to_add.pop(0)
            stops_by_date.append(current_stop)
        else:
            break

In [16]:
raw_df_W['stop'] = stops_by_date

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_70999/3492643979.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_df_W['stop'] = stops_by_date


In [17]:
df_with_stops = pd.concat([raw_df_E, raw_df_W], ignore_index=True)

In [17]:
df_with_stops = pd.concat([raw_df_E, raw_df_W], ignore_index=True)

In [18]:
df_with_stops.head()

,Unnamed: 0,Time,Vehicle,Gap,Headway,Schedule,Destination,day,day of the week,Riders after stop,stop
0,0,7:50:00AM (sched),4574.0,NaN,NaN,Scheduled at 7:50:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7
1,1,7:55:00AM (sched),4570.0,NaN,5:00,Scheduled at 7:55:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7
2,4,8:05:00AM (sched),4577.0,NaN,10:00,Scheduled at 8:05:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7
3,6,8:15:00AM (sched),4400.0,NaN,10:00,Scheduled at 8:15:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7
4,8,8:25:00AM (sched),4481.0,NaN,10:00,Scheduled at 8:25:00AM,East to Main Street Station,2025-01-01,2,NaN,Main Street Station at Bay 7


In [19]:
df_with_stops.to_csv('raw_with_stops.csv', index=False)